# Final model comparison — Improved Highway CNN vs. Standard CNN

**Notebook:** `06_final_model_comparison.ipynb`

**Research question:** Does the improved multi-stage Highway CNN outperform the same backbone built with ordinary Conv–BatchNorm–ReLU blocks?

**Reference from notebook 05:** Highway CNN — **88.80% validation accuracy**, **88.07% test accuracy**, **870,250 parameters**, trained for **20 epochs**. We will load the saved Highway checkpoint, train the Standard CNN under the same recipe, and compare **validation** metrics. No test-set evaluation is needed for choosing between models here.

This is a **same-backbone / same-width / same-depth** comparison, **not parameter matched**: the Highway blocks include extra gate convolutions. Results from one seed are exploratory, not proof of causality.

## 1. Experimental protocol

| Setting | Highway CNN (from notebook 05) | Standard CNN (new run) |
|---|---|---|
| Stages | 32 → 64 → 128 | 32 → 64 → 128 |
| Blocks per stage | 2 gated blocks | 2 Conv–BN–ReLU blocks |
| Initial convolution and downsampling | Identical | Identical |
| Head | Global average pool, dropout 0.2, linear | Same |
| Split | 45,000 train / 5,000 validation | Same |
| Augmentation | Random crop, horizontal flip | Same |
| Optimizer | AdamW | AdamW |
| Learning rate | 0.001 → cosine annealing to 0.00001 | Same |
| Weight decay | 0.0001 | Same |
| Gradient clipping | Max norm 5.0 | Same |
| Epochs | 20 | 20 |
| Random seed | 42 | 42 |

**Important:** This keeps the experimental protocol consistent, but it does not isolate gating from parameter count. Existing Highway results are reused from the checkpoint so they are not overwritten.

## 2. Setup and reproducibility

In [ ]:
from pathlib import Path
import sys, random, time, copy

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project:", PROJECT_ROOT)
print("Device:", device)
print("PyTorch:", torch.__version__)

## 3. Recreate the dataset split and loaders

The same indices and transforms as notebook 05 ensure the Standard CNN is trained and evaluated on exactly the same examples.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
BATCH_SIZE = 128
NUM_WORKERS = 0  # Notebook / Windows-friendly default

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_data_augmented = torchvision.datasets.CIFAR10(
    root=str(PROJECT_ROOT / "data"), train=True, download=True,
    transform=train_transform,
)
train_data_eval = torchvision.datasets.CIFAR10(
    root=str(PROJECT_ROOT / "data"), train=True, download=False,
    transform=eval_transform,
)
test_dataset = torchvision.datasets.CIFAR10(
    root=str(PROJECT_ROOT / "data"), train=False, download=True,
    transform=eval_transform,
)

indices = torch.randperm(50000, generator=torch.Generator().manual_seed(SEED)).tolist()
val_indices, train_indices = indices[:5000], indices[5000:]
assert set(train_indices).isdisjoint(val_indices)
train_subset = Subset(train_data_augmented, train_indices)
val_subset = Subset(train_data_eval, val_indices)

def make_train_loader(seed=SEED):
    return DataLoader(
        train_subset, batch_size=BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    )

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
class_names = test_dataset.classes
print("Train/val/test:", len(train_subset), len(val_subset), len(test_dataset))
print("Train/val/test batches:", len(make_train_loader()), len(val_loader), len(test_loader))

## 4. Define comparable architectures

The Standard block substitutes

$$H(x) = \operatorname{ReLU}(\operatorname{BN}(\operatorname{Conv}(x)))$$

for the Highway mixture

$$y = H(x)T(x)+x(1-T(x)).$$

Everything outside the six internal blocks (stem, stage widths, downsampling, head) is held constant. The Standard CNN has fewer trainable parameters because it has no gating convolutions.

In [ ]:
class ImprovedHighwayBlock(nn.Module):
    def __init__(self, channels, gate_bias=-3.0):
        super().__init__()
        self.transform = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=False),
        )
        self.gate = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        nn.init.constant_(self.gate.bias, gate_bias)

    def forward(self, x):
        transformed = self.transform(x)
        t = torch.sigmoid(self.gate(x))
        return transformed * t + x * (1.0 - t)


class ImprovedHighwayCNN(nn.Module):
    def __init__(self, num_classes=10, widths=(32, 64, 128),
                 blocks_per_stage=2, gate_bias=-3.0, dropout=0.2):
        super().__init__()
        w1, w2, w3 = widths
        self.stem = nn.Sequential(
            nn.Conv2d(3, w1, 3, padding=1, bias=False),
            nn.BatchNorm2d(w1),
            nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(*[
            ImprovedHighwayBlock(w1, gate_bias) for _ in range(blocks_per_stage)
        ])
        self.down1 = nn.Sequential(
            nn.Conv2d(w1, w2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(w2),
            nn.ReLU(inplace=True),
        )
        self.stage2 = nn.Sequential(*[
            ImprovedHighwayBlock(w2, gate_bias) for _ in range(blocks_per_stage)
        ])
        self.down2 = nn.Sequential(
            nn.Conv2d(w2, w3, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(w3),
            nn.ReLU(inplace=True),
        )
        self.stage3 = nn.Sequential(*[
            ImprovedHighwayBlock(w3, gate_bias) for _ in range(blocks_per_stage)
        ])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(w3, num_classes),
        )

    def forward(self, x):
        x = self.stage1(self.stem(x))
        x = self.stage2(self.down1(x))
        x = self.stage3(self.down2(x))
        return self.head(x)


MODEL_CONFIG = dict(num_classes=10, widths=(32, 64, 128),
                    blocks_per_stage=2, gate_bias=-3.0, dropout=0.2)

def new_model():
    return ImprovedHighwayCNN(**MODEL_CONFIG).to(device)

def count_parameters(net):
    return sum(p.numel() for p in net.parameters() if p.requires_grad)

In [ ]:
class StandardBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.transform = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=False),
        )

    def forward(self, x):
        return self.transform(x)


class ImprovedStandardCNN(nn.Module):
    def __init__(self, num_classes=10, widths=(32, 64, 128),
                 blocks_per_stage=2, dropout=0.2):
        super().__init__()
        w1, w2, w3 = widths
        self.stem = nn.Sequential(
            nn.Conv2d(3, w1, 3, padding=1, bias=False),
            nn.BatchNorm2d(w1),
            nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(*[
            StandardBlock(w1) for _ in range(blocks_per_stage)
        ])
        self.down1 = nn.Sequential(
            nn.Conv2d(w1, w2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(w2),
            nn.ReLU(inplace=True),
        )
        self.stage2 = nn.Sequential(*[
            StandardBlock(w2) for _ in range(blocks_per_stage)
        ])
        self.down2 = nn.Sequential(
            nn.Conv2d(w2, w3, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(w3),
            nn.ReLU(inplace=True),
        )
        self.stage3 = nn.Sequential(*[
            StandardBlock(w3) for _ in range(blocks_per_stage)
        ])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(w3, num_classes),
        )

    def forward(self, x):
        x = self.stage1(self.stem(x))
        x = self.stage2(self.down1(x))
        x = self.stage3(self.down2(x))
        return self.head(x)


STANDARD_CONFIG = dict(num_classes=10, widths=(32, 64, 128),
                       blocks_per_stage=2, dropout=0.2)

highway_reference = ImprovedHighwayCNN(**MODEL_CONFIG).to(device)
standard_model = ImprovedStandardCNN(**STANDARD_CONFIG).to(device)
print(f"Highway CNN parameters:  {count_parameters(highway_reference):,}")
print(f"Standard CNN parameters: {count_parameters(standard_model):,}")
assert count_parameters(highway_reference) == 870250, "Highway config differs from notebook 05"

## 5. Verify shapes, loss, and gradients

A quick smoke test ensures both models produce ten logits and support backpropagation before training.

In [ ]:
images, labels = next(iter(val_loader))
images, labels = images[:4].to(device), labels[:4].to(device)
smoke_criterion = nn.CrossEntropyLoss()
for name, net in (("Highway", highway_reference), ("Standard", standard_model)):
    net.train()
    net.zero_grad(set_to_none=True)
    logits = net(images)
    loss = smoke_criterion(logits, labels)
    loss.backward()
    assert logits.shape == (4, 10)
    assert torch.isfinite(loss).item()
    assert net.stem[0].weight.grad is not None
    print(f"{name:8s} | logits {tuple(logits.shape)} | loss {loss.item():.4f} | gradients OK")
print("Smoke test passed.")

## 6. Load the Highway reference checkpoint

This is the **model actually trained in notebook 05**, not an untrained copy. The path and config are verified; if the file is missing, stop and copy `improved_highway_v1_best.pth` from your existing project's `checkpoints/improved_highway/` folder.

The checkpoint's saved history contains the full learning curves. We compare **validation accuracy**, not the previously observed test result, to avoid further test-driven model selection.

In [ ]:
highway_path = PROJECT_ROOT / "checkpoints" / "improved_highway" / "improved_highway_v1_best.pth"
if not highway_path.exists():
    raise FileNotFoundError(f"Missing Highway checkpoint: {highway_path}")

highway_checkpoint = torch.load(highway_path, map_location="cpu", weights_only=True)
highway_config = highway_checkpoint["model_config"]
assert tuple(highway_config["widths"]) == (32, 64, 128)
assert highway_config["blocks_per_stage"] == 2

highway_model = ImprovedHighwayCNN(**highway_config).to(device)
highway_model.load_state_dict(highway_checkpoint["model_state_dict"])
highway_model.eval()
highway_history = highway_checkpoint["history"]
highway_best_epoch = highway_checkpoint["epoch"]

print("Loaded:", highway_path)
print("Highway selected epoch:", highway_best_epoch)
print("Highway best validation accuracy:", f"{max(highway_history['val_accuracy']):.2f}%")
print("Highway parameters:", f"{count_parameters(highway_model):,}")

## 7. Training helpers and configuration

Use the same training function as notebook 05, including gradient clipping. The Standard CNN is initialized freshly after setting the seed.

In [ ]:
EPOCHS = 20
LEARNING_RATE = 1e-3
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 5.0

criterion = nn.CrossEntropyLoss()

def run_epoch(net, loader, optimizer=None):
    training = optimizer is not None
    net.train(training)
    running_loss = 0.0
    correct, total = 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = net(images)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss).item():
                raise RuntimeError("Non-finite loss: stop and inspect training")
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    net.parameters(), max_norm=MAX_GRAD_NORM,
                    error_if_nonfinite=True,
                )
                optimizer.step()
            count = labels.size(0)
            running_loss += loss.item() * count
            correct += (logits.argmax(1) == labels).sum().item()
            total += count
    return running_loss / total, 100.0 * correct / total

## 8. Train the Standard CNN and save checkpoints

This cell trains **only the Standard CNN**, not the Highway CNN again. It writes checkpoints to `checkpoints/final_comparison/` and never overwrites notebook 05 files. The `latest` and `best` files update at the end of each completed epoch.

**Running this cell again restarts Standard CNN training from scratch.** If a run is interrupted, do not rerun without implementing checkpoint resumption.

In [ ]:
RUN_NAME = "standard_multistage_v1"
checkpoint_dir = PROJECT_ROOT / "checkpoints" / "final_comparison"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_path = checkpoint_dir / f"{RUN_NAME}_latest.pth"
best_path = checkpoint_dir / f"{RUN_NAME}_best.pth"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

net = ImprovedStandardCNN(**STANDARD_CONFIG).to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=MIN_LR
)
train_loader = make_train_loader(seed=SEED)
standard_history = {"train_loss": [], "train_accuracy": [],
                    "val_loss": [], "val_accuracy": [], "lr": []}
best_val_acc, best_epoch = float("-inf"), None
started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    lr_used = optimizer.param_groups[0]["lr"]
    tr_loss, tr_acc = run_epoch(net, train_loader, optimizer)
    va_loss, va_acc = run_epoch(net, val_loader)
    scheduler.step()

    for key, value in {"train_loss": tr_loss, "train_accuracy": tr_acc,
                       "val_loss": va_loss, "val_accuracy": va_acc,
                       "lr": lr_used}.items():
        standard_history[key].append(value)

    payload = {
        "epoch": epoch,
        "model_state_dict": net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": standard_history,
        "model_config": STANDARD_CONFIG,
        "seed": SEED,
        "run_name": RUN_NAME,
    }
    if va_acc > best_val_acc:
        best_val_acc, best_epoch = va_acc, epoch
        torch.save(payload, best_path)
    torch.save(payload, latest_path)

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.2f}% | "
          f"val loss {va_loss:.4f} acc {va_acc:.2f}% | "
          f"LR {lr_used:.6f} | best {best_val_acc:.2f}% (epoch {best_epoch})",
          flush=True)

print(f"Standard CNN training complete in {(time.perf_counter()-started)/60:.1f} min")
print("Best Standard validation accuracy:", f"{best_val_acc:.2f}%", "at epoch", best_epoch)
print("Best checkpoint:", best_path)

## 9. Compare validation results

We compare the **best validation checkpoints**, which is the model-selection rule used in notebook 05. We also compare final-epoch curves, model sizes, and loss trends. No test-set evaluations are used here.

In [ ]:
standard_checkpoint = torch.load(best_path, map_location="cpu", weights_only=True)
standard_history = standard_checkpoint["history"]
standard_best_epoch = standard_checkpoint["epoch"]

highway_best_val = max(highway_history["val_accuracy"])
standard_best_val = max(standard_history["val_accuracy"])

print(f"{'Metric':<32} {'Highway CNN':>15} {'Standard CNN':>15}")
print("-" * 66)
rows = [
    ("Parameters", count_parameters(highway_model),
     count_parameters(ImprovedStandardCNN(**STANDARD_CONFIG))),
    ("Best validation epoch", highway_best_epoch, standard_best_epoch),
    ("Best validation accuracy (%)", f"{highway_best_val:.2f}",
     f"{standard_best_val:.2f}"),
    ("Final epoch val accuracy (%)", f"{highway_history['val_accuracy'][-1]:.2f}",
     f"{standard_history['val_accuracy'][-1]:.2f}"),
    ("Selected-epoch validation loss", f"{highway_history['val_loss'][highway_best_epoch-1]:.4f}",
     f"{standard_history['val_loss'][standard_best_epoch-1]:.4f}"),
]
for metric, highway, standard in rows:
    print(f"{metric:<32} {str(highway):>15} {str(standard):>15}")
print(f"\nHighway advantage: {highway_best_val - standard_best_val:+.2f} percentage points")

In [ ]:
for metric, ylabel in (("val_accuracy", "Validation accuracy (%)"),
                       ("val_loss", "Validation cross-entropy loss"),
                       ("train_accuracy", "Training accuracy (%)")):
    plt.figure(figsize=(9, 4.5))
    plt.plot(range(1, len(highway_history[metric]) + 1),
             highway_history[metric], marker="o", label="Highway CNN")
    plt.plot(range(1, len(standard_history[metric]) + 1),
             standard_history[metric], marker="o", label="Standard CNN")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"Improved CNN comparison — {ylabel}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## 10. Final analysis and conclusions

**Reference: Improved Highway CNN (notebook 05)**

- 870,250 parameters; 20 epochs
- Best validation accuracy: **88.80%** (epoch 20)
- Previously measured test accuracy: **88.07%** (reported for context, **not used to select models**)

**Fill after Section 9:**

| Metric | Highway CNN | Standard CNN |
|---|---:|---:|
| Parameters | 870,250 | _pending_ |
| Best validation accuracy | 88.80% | _pending_ |
| Best validation epoch | 20 | _pending_ |
| Selected-epoch validation loss | 0.3456 | _pending_ |
| Final training accuracy | 93.30% | _pending_ |

### Interpretation questions

1. Which architecture has higher best validation accuracy?
2. Does either model converge faster in the first five epochs?
3. How much additional capacity do Highway gates require?
4. Do the results demonstrate an advantage from gating itself? **Not conclusively**: parameter counts differ and this is a single-seed comparison.

### Final project conclusion

Our experiments tested both the **optimization motivation** of Highway Networks (notebook 04) and their **classification performance** (notebooks 05–06). The depth experiments showed substantial optimization benefits at 20–50 blocks in our simplified setup, alongside instability at 100 blocks. The improved multi-stage Highway CNN achieved 88.07% CIFAR-10 test accuracy. This final validation comparison tests whether the upgraded training recipe and backbone also perform well without gates.

**Scope:** An educational implementation and controlled exploratory comparison—not an exact reproduction of the original paper's architecture or a statistically conclusive study. For stronger evidence, repeat with multiple seeds and a parameter-matched baseline.